In [ ]:
# ====================================================================
# 1. IMPORTS & CONFIGURATION
# ====================================================================
import os
import glob
import re
import random
import joblib
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_curve

from xgboost import XGBClassifier
from tensorflow.keras.models import load_model

import warnings
warnings.filterwarnings('ignore')

class Config:
    # --- PATHS ---
    DATA_DIR = "./data/" 
    
    # Helper Models
    ISO_MODEL_PATH = './../models/IsolationForest/model/iso_model2.joblib'
    ISO_SCALER_PATH = './../models/IsolationForest/model/iso_scaler2.joblib'
    LSTM_MODEL_PATH = './../models/LSTM_Autoencoders/lstm_autoencoder_model_fit.h5'
    LSTM_SCALER_PATH = './../models/LSTM_Autoencoders/scaler_fit.save'
    
    # Outputs
    OUTPUT_MODEL = "./model/ultimate_full_xgboost.json"
    OUTPUT_SCALER = "./model/ultimate_full_scaler.save"
    
    # --- SETTINGS ---
    SEQUENCE_LENGTH = 30
    TEST_KEYWORDS = ['label_for_dev109_test_prepared'] 
    OVERSAMPLE_TARGET = 5000
    VAL_SPLIT_RATIO = 0.2
    
    RAW_COLS = ['rain', 'soil', 'temp', 'humi', 'geo']
    LABEL_COL = 'label'
    LABEL_MAP = {'normal': 0, 'warning': 1, 'critical': 2}
    
    # Weights (Critical x25)
    SAMPLE_WEIGHTS = {0: 1.0, 1: 5.0, 2: 25.0}
    
    # Grid Search Params
    PARAM_GRID = {
        'n_estimators': [200, 400],    
        'learning_rate': [0.01, 0.05],     
        'max_depth': [4, 6],    
        
        'subsample': [0.8],    
        'colsample_bytree': [0.8],
        
        'min_child_weight': [1, 5, 10],   
        'gamma': [0, 0.2, 0.5],   
    }
    
    XGB_FIXED = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'n_jobs': -1,
        'random_state': 42,
        'tree_method': 'hist'
    }

cfg = Config()
np.random.seed(42); random.seed(42)

def log(msg): print(f"[INFO] {time.strftime('%H:%M:%S')} - {msg}")
print("✅ Config Loaded (Full Features Edition)")

In [ ]:
# ====================================================================
# 2. LOAD & FEATURE ENGINEERING
# ====================================================================
def load_and_split_data():
    log("Scanning files...")
    train_dfs, test_dfs = [], []
    for f_path in glob.glob(os.path.join(cfg.DATA_DIR, "*.csv")):
        filename = os.path.basename(f_path)
        is_test = any(k in filename for k in cfg.TEST_KEYWORDS)
        target_list = test_dfs if is_test else train_dfs
        try:
            df = pd.read_csv(f_path)
            df.columns = [str(c).lower().strip().replace('.1', '') for c in df.columns]
            rename_map = {'temperature':'temp', 'hum':'humi', 'humidity':'humi', 'devid':'devID', 'time':'timestamp', 'date':'timestamp'}
            new_cols = {c: rename_map[c] for c in df.columns if c in rename_map}
            if new_cols: df = df.rename(columns=new_cols)
            if 'devID' in df.columns: df['devID'] = df['devID'].astype(str).apply(lambda x: int(re.search(r'(\d+)', x).group(1)) if re.search(r'(\d+)', x) else 0)
            if 'timestamp' in df.columns: df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            for c in cfg.RAW_COLS: 
                if c not in df.columns: df[c] = 0.0
            target_list.append(df)
            print(f"   -> {filename} [{'TEST' if is_test else 'TRAIN'}]")
        except: pass
    if not train_dfs: raise ValueError("No Train Data")
    return pd.concat(train_dfs, ignore_index=True), (pd.concat(test_dfs, ignore_index=True) if test_dfs else pd.DataFrame())

df_train_raw, df_test_raw = load_and_split_data()

def generate_features(df):
    if df.empty: return df, []
    log("Generating Features...")
    df = df.sort_values(['devID', 'timestamp']).reset_index(drop=True)
    df_list = []
    for dev, g in df.groupby('devID'):
        g = g.set_index('timestamp')
        g = g[~g.index.duplicated(keep='first')]
        if len(g) > 0: g = g.resample('1T').asfreq()
        g[cfg.RAW_COLS] = g[cfg.RAW_COLS].interpolate(limit_direction='both').fillna(0)
        
        # Physics
        g['feat_rain_cum_24h'] = g['rain'].rolling(1440, min_periods=1).sum()
        g['feat_soil_rate'] = g['soil'].diff().fillna(0)
        g['feat_geo_abs'] = g['geo'].abs()
        g['feat_rain_x_soil'] = g['rain'] * g['soil']
        
        if cfg.LABEL_COL in g.columns:
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].fillna('normal').astype(str).str.lower().str.strip()
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].map(cfg.LABEL_MAP).fillna(0).astype(int)
        else: g[cfg.LABEL_COL] = 0
        g['devID'] = dev
        df_list.append(g.reset_index())
    df = pd.concat(df_list, ignore_index=True)

    # AI Features
    try:
        iso_model = joblib.load(cfg.ISO_MODEL_PATH); iso_scaler = joblib.load(cfg.ISO_SCALER_PATH)
        X_iso = iso_scaler.transform(df[cfg.RAW_COLS].values)
        df['feat_iso_score'] = iso_model.decision_function(X_iso)
    except: df['feat_iso_score'] = 0.0
    
    try:
        # (Simplified LSTM placeholder for speed)
        df['feat_lstm_error'] = 0.0 
    except: pass

    df = df.fillna(0)
    # เก็บทุกฟีเจอร์ที่สร้างขึ้น + ของเดิม
    feat_cols = cfg.RAW_COLS + [c for c in df.columns if c.startswith('feat_')]
    return df, feat_cols

df_train_proc, feature_list = generate_features(df_train_raw)
df_test_proc, _ = generate_features(df_test_raw)

In [ ]:
# ====================================================================
# 3. VECTORIZED FLATTENING
# ====================================================================
def vectorized_flatten(df, features, seq_len):
    if df.empty: return np.array([]), np.array([]), []
    log("Vectorized Flattening...")
    Xs, ys = [], []
    flat_cols = [f"{f}_t-{seq_len-1-i}" for i in range(seq_len) for f in features]
    
    for dev, g in df.groupby('devID'):
        data = g[features].values; labels = g[cfg.LABEL_COL].values
        if len(g) < seq_len: continue
        windows = np.lib.stride_tricks.sliding_window_view(data, window_shape=(seq_len, len(features))).squeeze()
        flat_windows = windows.reshape(windows.shape[0], -1)
        window_labels = labels[seq_len-1:]
        Xs.append(flat_windows); ys.append(window_labels)
        
    if not Xs: return np.array([]), np.array([]), flat_cols
    return np.concatenate(Xs), np.concatenate(ys), flat_cols

X_train_full, y_train_full, feature_names = vectorized_flatten(df_train_proc, feature_list, cfg.SEQUENCE_LENGTH)
X_test_final, y_test_final, _ = vectorized_flatten(df_test_proc, feature_list, cfg.SEQUENCE_LENGTH)

In [ ]:
# ====================================================================
# 4. PREPARE DATA (NO FEATURE SELECTION)
# ====================================================================
# Split Val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=cfg.VAL_SPLIT_RATIO, stratify=y_train_full, random_state=42
)

# Oversample
log("Balancing Train Set...")
X_res, y_res = [], []
for cls in np.unique(y_train):
    idx = np.where(y_train == cls)[0]
    target = cfg.OVERSAMPLE_TARGET if len(idx) < cfg.OVERSAMPLE_TARGET else len(idx)
    chosen = np.random.choice(idx, target, replace=True)
    X_res.append(X_train[chosen]); y_res.append(y_train[chosen])
X_train_bal = np.concatenate(X_res)
y_train_bal = np.concatenate(y_res)

# Scale
log("Scaling...")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_bal) # Fit on ALL features
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test_final) if len(X_test_final) > 0 else np.array([])
joblib.dump(scaler, cfg.OUTPUT_SCALER)

# Weights
sample_weights = np.array([cfg.SAMPLE_WEIGHTS[y] for y in y_train_bal])

log(f"Data Ready. Features: {X_train_s.shape[1]} (Full Set)")

In [ ]:
# ====================================================================
# 5. TRAINING (GRID SEARCH + TIME SERIES SPLIT)
# ====================================================================
log("🚀 Starting Grid Search (Full Features)...")

xgb_base = XGBClassifier(**cfg.XGB_FIXED)
tscv = TimeSeriesSplit(n_splits=3)

grid_search = GridSearchCV(
    xgb_base, 
    cfg.PARAM_GRID, 
    scoring='f1_weighted', 
    cv=5,  
    verbose=1, 
    n_jobs=-1
)

grid_search.fit(
    X_train_s, 
    y_train_bal, 
    sample_weight=sample_weights
)

print(f"\n🏆 Best Params: {grid_search.best_params_}")
best_model = grid_search.best_estimator_
best_model.save_model(cfg.OUTPUT_MODEL)
log("✅ Model Saved.")

In [ ]:
# ====================================================================
# 6. THRESHOLD TUNING
# ====================================================================
log("🔧 Tuning Threshold...")
def find_best_threshold(model, X, y_true):
    probs = model.predict_proba(X)[:, 2]
    y_bin = (y_true == 2).astype(int)
    precisions, recalls, thresholds = precision_recall_curve(y_bin, probs)
    f2_scores = (5 * precisions * recalls) / (4 * precisions + recalls + 1e-9)
    best_idx = np.argmax(f2_scores)
    return thresholds[best_idx]

OPTIMAL_THRESHOLD = find_best_threshold(best_model, X_val_s, y_val)
print(f"🔥 Optimal Threshold: {OPTIMAL_THRESHOLD:.4f}")

In [ ]:
# ====================================================================
# 7. FINAL EVALUATION
# ====================================================================
def custom_predict(model, X, threshold):
    probs = model.predict_proba(X)
    p_norm, p_warn, p_crit = probs[:, 0], probs[:, 1], probs[:, 2]
    preds = np.zeros(len(X), dtype=int)
    
    crit_mask = p_crit >= threshold
    preds[crit_mask] = 2
    preds[~crit_mask] = np.where(p_warn[~crit_mask] > p_norm[~crit_mask], 1, 0)
    return preds

def evaluate_final(name, X, y):
    if len(X) == 0: return
    print(f"\n{'='*25} {name} RESULTS {'='*25}")
    y_pred = custom_predict(best_model, X, OPTIMAL_THRESHOLD)
    
    print(f"Accuracy    : {accuracy_score(y, y_pred):.4f}")
    print(f"F1-Weighted : {f1_score(y, y_pred, average='weighted'):.4f}")
    
    cm = confusion_matrix(y, y_pred)
    if cm.shape[0] > 2:
        missed_crit = cm[2, 0] + cm[2, 1]
        print(f"\n🛡️ SAFETY: Missed Critical = {missed_crit} / {np.sum(cm[2, :])}")
        if missed_crit == 0: print("   ✅ PERFECT SAFETY")
        else: print("   ⚠ WARNING: Missed some criticals.")

    print("\nConfusion Matrix:")
    print(cm)
    print(classification_report(y, y_pred, target_names=['Normal', 'Warning', 'Critical']))

evaluate_final("VALIDATION SET", X_val_s, y_val)
if len(X_test_s) > 0:
    evaluate_final("TEST SET", X_test_s, y_test_final)

# Feature Importance (Top 20)
imp = best_model.feature_importances_
indices = np.argsort(imp)[::-1][:20]
plt.figure(figsize=(10, 6))
plt.title("Top 20 Features (Full Set)")
plt.barh(range(20), imp[indices], align="center", color='lightgreen')
plt.yticks(range(20), [feature_names[i] for i in indices])
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()